In [ ]:
%pip install pandas
import pandas as pd
df = pd.read_csv("fake_job_postings.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df['fraudulent'].value_counts()

In [ ]:
import matplotlib.pyplot as plt

df['fraudulent'].value_counts().plot(kind='bar')
plt.title("Fraudulent vs Real Job Postings")
plt.xlabel("Fraudulent")
plt.ylabel("Count")
plt.show()


In [ ]:
df = df.drop(columns=['job_id'])

In [ ]:
df['description'].str.len().describe()

In [ ]:
df.isnull().mean().sort_values(ascending=False)

In [ ]:
df = df.drop(columns=['salary_range', 'department'])
df.shape

In [ ]:
df['employment_type'].value_counts(dropna=False).head(10)

In [ ]:
df.groupby('fraudulent')[['has_company_logo', 'telecommuting', 'has_questions']].mean()

In [ ]:
df.groupby('fraudulent')['has_company_logo'].mean().plot(kind='bar')
plt.title("Company Logo Presence vs Fraud")
plt.ylabel("Average Logo Presence")
plt.show()

In [ ]:
df = df.drop(columns=['location'])

In [ ]:
df[['description', 'requirements', 'company_profile']].isnull().mean()

In [ ]:
text_cols = ['description', 'requirements', 'company_profile', 'benefits', 'title']
df[text_cols] = df[text_cols].fillna("")

In [ ]:
df['desc_len'] = df['description'].str.len()
df['req_len'] = df['requirements'].str.len()
df['profile_len'] = df['company_profile'].str.len()
df['benefits_len'] = df['benefits'].str.len()

df[['desc_len', 'req_len', 'profile_len', 'benefits_len']].describe()

In [ ]:
text_cols = ['title', 'description', 'requirements', 'company_profile', 'benefits']
cat_cols = ['employment_type', 'required_experience', 'required_education', 'industry', 'function']
num_cols = [
    'has_company_logo',
    'has_questions',
    'telecommuting',
    'desc_len',
    'req_len',
    'profile_len',
    'benefits_len'
]
print("Text cols",text_cols)
print("Cat cols",cat_cols)
print("Num cols",num_cols)


In [ ]:
df[text_cols] = df[text_cols].fillna("")

In [ ]:
df[cat_cols] = df[cat_cols].fillna("Unknown")

In [ ]:
from sklearn.model_selection import train_test_split

X = df[text_cols + cat_cols + num_cols]
y = df['fraudulent']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train.shape, X_test.shape


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

text_feature = 'description'

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(stop_words='english', max_features=5000), text_feature),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('numeric', 'passthrough', num_cols)
    ]
)

model = Pipeline([
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=200))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))


In [ ]:
df['text_all'] = (
    df['title'] + " " +
    df['description'] + " " +
    df['requirements'] + " " +
    df['company_profile'] + " " +
    df['benefits']
)

X = df[['text_all'] + cat_cols + num_cols]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train.shape, X_test.shape

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(stop_words='english', max_features=10000), 'text_all'),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('numeric', 'passthrough', num_cols)
    ]
)

In [ ]:
model = Pipeline([
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=200))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

model_svm = Pipeline([
    ('preprocess', preprocessor),
    ('clf', LinearSVC(class_weight='balanced'))
])

model_svm.fit(X_train, y_train)

y_pred_svm = model_svm.predict(X_test)

print(classification_report(y_test, y_pred_svm))


In [ ]:
df.info()

In [ ]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)

X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

X_train_res.shape, y_train_res.value_counts()

In [ ]:
model_svm_resampled = Pipeline([
    ('preprocess', preprocessor),
    ('clf', LinearSVC())
])

model_svm_resampled.fit(X_train_res, y_train_res)

y_pred_res = model_svm_resampled.predict(X_test)

print(classification_report(y_test, y_pred_res))


In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.svm import LinearSVC
# from sklearn.metrics import classification_report
# from imblearn.over_sampling import SMOTE

# X_num_cat = df[cat_cols + num_cols]
# y = df['fraudulent']

# X_train_nc, X_test_nc, y_train_nc, y_test_nc = train_test_split(
#     X_num_cat, y, test_size=0.2, stratify=y, random_state=42
# )

# ohe = OneHotEncoder(handle_unknown='ignore')

# X_train_enc = ohe.fit_transform(X_train_nc)
# X_test_enc  = ohe.transform(X_test_nc)

# smote = SMOTE(random_state=42)
# X_train_smote, y_train_smote = smote.fit_resample(X_train_enc, y_train_nc)

# print("Shapes after SMOTE:", X_train_smote.shape, y_train_smote.value_counts())

# clf_smote = LinearSVC()
# clf_smote.fit(X_train_smote, y_train_smote)

# y_pred_smote = clf_smote.predict(X_test_enc)

# print(classification_report(y_test_nc, y_pred_smote))


dont take smote

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from imblearn.over_sampling import RandomOverSampler

X = df[['text_all'] + cat_cols + num_cols]
y = df['fraudulent']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(stop_words='english', max_features=10000), 'text_all'),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('numeric', 'passthrough', num_cols)
    ]
)

final_model = Pipeline([
    ('preprocess', preprocessor),
    ('clf', LinearSVC())
])

final_model.fit(X_train_res, y_train_res)

y_pred = final_model.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
model.decision_function(X_new)[0]

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm_display = ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=[0, 1]
)
plt.title("Confusion Matrix - Final SVM Fraud Detector")
plt.show()

In [ ]:
# def predict_job(text_title, text_description, text_requirements="", text_company_profile="", text_benefits="",
#                 employment_type="Unknown", required_experience="Unknown", required_education="Unknown",
#                 industry="Unknown", function="Unknown", has_company_logo=1, has_questions=0, telecommuting=0):
    
#     text_all = " ".join([
#         text_title,
#         text_description,
#         text_requirements,
#         text_company_profile,
#         text_benefits
#     ])
    
#     data = {
#         'text_all': [text_all],
#         'employment_type': [employment_type],
#         'required_experience': [required_experience],
#         'required_education': [required_education],
#         'industry': [industry],
#         'function': [function],
#         'has_company_logo': [has_company_logo],
#         'has_questions': [has_questions],
#         'telecommuting': [telecommuting]
#     }
    
#     import pandas as pd
#     X_new = pd.DataFrame(data)
    
#     pred = final_model.predict(X_new)[0]
#     prob_label = "Fraudulent" if pred == 1 else "Real"
#     print("Prediction:", prob_label)


In [ ]:
# predict_job(
#     text_title="Data Entry Clerk - Work From Home",
#     text_description="No experience needed, make $5000 per week, limited spots, apply now!!!",
#     has_company_logo=0,
#     has_questions=0,
#     telecommuting=1
# )


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(final_model, X, y, cv=5, scoring='f1_macro')
scores, scores.mean()


In [ ]:
import joblib

joblib.dump(final_model, "fraud_job_model.pkl")

joblib.dump({
    "cat_cols": cat_cols,
    "num_cols": num_cols
}, "fraud_job_meta.pkl")

print("Model saved!")

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2
import json

# 1. Load data (or reuse df if already loaded)
df = pd.read_csv("fake_job_postings.csv")

# Basic text preprocessing: combine text
text_cols = ["title", "description", "requirements", "company_profile", "benefits"]
for c in text_cols:
    df[c] = df[c].fillna("")

df["text_all"] = (
    df["title"] + " " +
    df["description"] + " " +
    df["requirements"] + " " +
    df["company_profile"] + " " +
    df["benefits"]
)

X_text = df["text_all"]
y = df["fraudulent"]

# 2. Vectorize with n-grams (unigrams + bigrams)
vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5  # ignore super-rare phrases
)
X_vec = vectorizer.fit_transform(X_text)

# 3. Use chi2 to find terms strongly associated with fraud (label 1)
chi2_scores, p_values = chi2(X_vec, y)
feature_names = vectorizer.get_feature_names_out()

# 4. Get top N fraud-associated phrases
import numpy as np

N = 50  # number of top red-flag ngrams you want
indices = np.argsort(chi2_scores)[::-1][:N]
top_terms = [feature_names[i] for i in indices]

top_terms[:30]


In [ ]:
red_flags_dict = {
    "auto_red_flags": top_terms
}

with open("red_flags.json", "w", encoding="utf-8") as f:
    json.dump(red_flags_dict, f, indent=2, ensure_ascii=False)

print("Saved red_flags.json")
